# Raw Sequences (By Raw)

将原始序列转换为一维数组。

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 输入文件路径
data_dir = Path('../../../data')
timeline_file = data_dir / 'pickle/timeline.pkl'

# 我们引入 performance.csv/responses.csv 作为骨架，以防 timeline 使得零操作的 Trial 彻底丢失
performance_file = data_dir / 'analysis/performance/performance.csv'

# 输出文件路径
output_dir = Path('output')
raw_sequences_file = output_dir / 'raw_sequences.csv'
output_dir.mkdir(exist_ok=True)

In [ ]:
# 加载数据
timeline = pd.read_pickle(timeline_file)
performance = pd.read_csv(performance_file)

## 聚合数据

In [ ]:
# 原始序列处理为一维数组

def build_raw_sequence(trial_timeline, duration=300):
    """将单个 trial 的时间线压缩为固定长度的一维数组。"""
    sequence = np.zeros(duration, dtype=int)
    
    if trial_timeline is None or trial_timeline.empty:
        return sequence.tolist()

    ai_events = trial_timeline.loc[trial_timeline['type'] == 'ai_click', 'time'].copy()
    if ai_events.empty:
        return sequence.tolist()

    ai_events = pd.to_numeric(ai_events, errors='coerce').dropna()
    if ai_events.empty:
        return sequence.tolist()

    # 兼容毫秒级时间戳：若数值明显大于 300 秒，则转换为秒
    if ai_events.abs().max() > 1000:
        ai_events = ai_events / 1000.0

    for event_time in ai_events:
        second = int(np.floor(event_time))
        if 0 <= second < duration:
            sequence[second] = 1

    return sequence.tolist()

# 使用 performance 做基准骨架，确保没有缺失的数据点
raw_sequence_rows = []

if not performance.empty:
    for _, row in performance.iterrows():
        pid = row['participant_id']
        item = row['item_name']
        trial = row['trial_index']
        
        # 提取 timeline 中对应的片段
        mask = (timeline['participant_id'] == pid) & \
               (timeline['item_name'] == item) & \
               (timeline['trial_index'] == trial)
        trial_timeline = timeline[mask]
        
        raw_sequence = build_raw_sequence(trial_timeline)
        raw_sequence_rows.append({
            'participant_id': pid,
            'item_name': item,
            'trial_index': trial,
            'raw_sequence': json.dumps(raw_sequence, ensure_ascii=False),
        })

raw_sequences = pd.DataFrame(raw_sequence_rows)
print('raw_sequences 形状:', raw_sequences.shape)
if not raw_sequences.empty:
    print(raw_sequences.head(1).to_dict(orient='records')[0])

raw_sequences.to_csv(raw_sequences_file, index=False, encoding='utf-8-sig')
print('raw_sequences 已保存，raw_sequences.csv ->', raw_sequences_file)